# Visualização Interativa: Distribuição Geográfica de Armadores de Pesca no Brasil

**Disciplina:** Computação Gráfica e Visualização I (INF01047) — INF/UFRGS  
**Laboratório 3**

---

**Dados:** Registros de Armadores de Pesca – MPA/SERMOP  
**Fonte:** https://agromapa-my.sharepoint.com/:x:/g/personal/lucas_ramos_mpa_gov_br/ERtMMkNNT1JDtoLNM65A_ZQBfbRer6f3CJUo3GsBfiV9xQ

## 1. Instalação de dependências

In [ ]:
# Execute esta célula para instalar as bibliotecas necessárias
# (No Google Colab já estão disponíveis, mas a instalação é idempotente)
!pip install folium pandas openpyxl geopandas requests --quiet

## 2. Importações

In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap, MarkerCluster
import json
import requests
import warnings
warnings.filterwarnings('ignore')

print('Bibliotecas importadas com sucesso!')

## 3. Carregamento dos dados

> **Atenção:** Baixe o arquivo Excel do link do README (banco de dados MPA) e faça upload no Colab (ou coloque na mesma pasta do notebook). Renomeie-o como `armadores_pesca.xlsx`.
>
> Se preferir testar sem o arquivo real, a célula abaixo gera dados sintéticos baseados em distribuições reais para demonstração.

In [ ]:
import os
import numpy as np

# -------------------------------------------------------------------
# OPÇÃO A: Carregamento do arquivo real
# Descomente e ajuste o caminho se tiver o arquivo
# -------------------------------------------------------------------
# df_raw = pd.read_excel('armadores_pesca.xlsx')
# print(df_raw.columns.tolist())  # Verifique os nomes das colunas
# print(df_raw.head())

# -------------------------------------------------------------------
# OPÇÃO B: Dados sintéticos realistas para demonstração
# (Baseados na distribuição geográfica real de pescadores no Brasil)
# Remove esta seção quando usar o arquivo real
# -------------------------------------------------------------------

np.random.seed(42)

# Coordenadas aproximadas dos principais municípios pesqueiros
# por estado com pesos realistas de concentração
municipios_pesca = [
    # Estado, Município, Lat, Lon, Peso (concentração relativa)
    # Região Norte
    ('PA', 'Belém',          -1.455, -48.502, 320),
    ('PA', 'Santarém',       -2.444, -54.708, 210),
    ('PA', 'Marabá',         -5.368, -49.117,  80),
    ('PA', 'Castanhal',      -1.295, -47.924,  95),
    ('AM', 'Manaus',         -3.119, -60.021, 280),
    ('AM', 'Itacoatiara',    -3.143, -58.444,  90),
    ('AM', 'Parintins',      -2.627, -56.735, 130),
    ('RO', 'Porto Velho',    -8.761, -63.900,  60),
    ('RO', 'Ji-Paraná',      -10.879,-61.949,  40),
    ('AC', 'Rio Branco',     -9.974, -67.808,  50),
    ('AP', 'Macapá',          0.035, -51.066, 110),
    ('AP', 'Santana',        -0.059, -51.182,  80),
    ('TO', 'Palmas',        -10.249, -48.324,  35),
    ('RR', 'Boa Vista',       2.820, -60.673,  30),
    # Região Nordeste
    ('MA', 'São Luís',       -2.529, -44.302, 340),
    ('MA', 'Imperatriz',     -5.526, -47.491, 110),
    ('MA', 'Cururupu',       -1.827, -44.869, 150),
    ('CE', 'Fortaleza',      -3.717, -38.543, 480),
    ('CE', 'Sobral',         -3.689, -40.349,  90),
    ('CE', 'Acaraú',         -2.885, -40.120, 170),
    ('CE', 'Camocim',        -2.902, -40.841, 200),
    ('RN', 'Natal',          -5.793, -35.200, 290),
    ('RN', 'Mossoró',        -5.187, -37.344, 110),
    ('RN', 'Areia Branca',   -4.950, -37.117, 180),
    ('PB', 'João Pessoa',    -7.119, -34.845, 220),
    ('PE', 'Recife',         -8.057, -34.882, 310),
    ('PE', 'Olinda',         -7.999, -34.855, 140),
    ('PE', 'Cabo de Sto Agostinho',-8.289,-35.032,120),
    ('AL', 'Maceió',         -9.666, -35.735, 200),
    ('SE', 'Aracaju',       -10.909, -37.071, 170),
    ('BA', 'Salvador',      -12.971, -38.511, 380),
    ('BA', 'Ilhéus',        -14.791, -39.033, 160),
    ('BA', 'Porto Seguro',  -16.430, -39.065, 140),
    ('BA', 'Valença',       -13.370, -39.078, 110),
    ('PI', 'Teresina',       -5.089, -42.802,  60),
    ('PI', 'Parnaíba',       -2.905, -41.776, 200),
    # Região Centro-Oeste
    ('MT', 'Cuiabá',        -15.596, -56.096,  80),
    ('MT', 'Várzea Grande',  -15.647,-56.131,   55),
    ('GO', 'Goiânia',       -16.686, -49.264,  65),
    ('MS', 'Campo Grande',  -20.469, -54.620,  70),
    ('MS', 'Corumbá',       -19.008, -57.654, 120),
    ('DF', 'Brasília',      -15.780, -47.929,  40),
    # Região Sudeste
    ('SP', 'Santos',        -23.960, -46.333, 250),
    ('SP', 'São Paulo',     -23.549, -46.633, 180),
    ('SP', 'Guarujá',       -23.993, -46.256, 200),
    ('SP', 'São Sebastião', -23.800, -45.407, 150),
    ('SP', 'Ubatuba',       -23.434, -45.071, 130),
    ('RJ', 'Rio de Janeiro',-22.906, -43.172, 400),
    ('RJ', 'Niterói',       -22.883, -43.104, 180),
    ('RJ', 'Angra dos Reis',-23.007, -44.317, 160),
    ('RJ', 'Macaé',         -22.370, -41.786, 210),
    ('RJ', 'Cabo Frio',     -22.880, -42.019, 190),
    ('ES', 'Vitória',       -20.319, -40.338, 200),
    ('ES', 'Vila Velha',    -20.329, -40.292, 140),
    ('MG', 'Belo Horizonte',-19.919, -43.938,  60),
    # Região Sul
    ('SC', 'Florianópolis', -27.594, -48.548, 290),
    ('SC', 'Itajaí',        -26.906, -48.661, 350),
    ('SC', 'Navegantes',    -26.900, -48.654, 200),
    ('SC', 'São Francisco do Sul',-26.241,-48.632,180),
    ('PR', 'Paranaguá',     -25.520, -48.508, 240),
    ('PR', 'Guaratuba',     -25.882, -48.575, 130),
    ('PR', 'Pontal do Paraná',-25.611,-48.517,110),
    ('RS', 'Rio Grande',    -32.035, -52.098, 310),
    ('RS', 'Porto Alegre',  -30.033, -51.230, 150),
    ('RS', 'Pelotas',       -31.771, -52.342, 130),
    ('RS', 'Torres',        -29.333, -49.729, 100),
]

# Gerar linhas do dataframe com dispersão geográfica
rows = []
for estado, municipio, lat, lon, n in municipios_pesca:
    for _ in range(n):
        rows.append({
            'estado': estado,
            'municipio': municipio,
            'latitude': lat + np.random.normal(0, 0.18),
            'longitude': lon + np.random.normal(0, 0.18),
        })

df = pd.DataFrame(rows)

# Mapeamento estado → região
regioes_map = {
    'AC':'Norte','AM':'Norte','AP':'Norte','PA':'Norte',
    'RO':'Norte','RR':'Norte','TO':'Norte',
    'AL':'Nordeste','BA':'Nordeste','CE':'Nordeste','MA':'Nordeste',
    'PB':'Nordeste','PE':'Nordeste','PI':'Nordeste','RN':'Nordeste','SE':'Nordeste',
    'DF':'Centro-Oeste','GO':'Centro-Oeste','MS':'Centro-Oeste','MT':'Centro-Oeste',
    'ES':'Sudeste','MG':'Sudeste','RJ':'Sudeste','SP':'Sudeste',
    'PR':'Sul','RS':'Sul','SC':'Sul',
}
df['regiao'] = df['estado'].map(regioes_map)

print(f'Total de registros: {len(df)}')
print(df.groupby('regiao').size().rename('pescadores').sort_values(ascending=False))

---
## 4. Se você tem o arquivo real: adapte aqui

Se carregou `armadores_pesca.xlsx`, execute a célula abaixo para mapear as colunas reais do dataset para o formato que o código espera (`estado`, `municipio`, `latitude`, `longitude`).

> Verifique quais colunas estão disponíveis com `df_raw.columns.tolist()` e ajuste o mapeamento conforme necessário.

In [ ]:
# =========================================================
# DESCOMENTE E ADAPTE quando usar o arquivo real
# =========================================================

# df = df_raw.rename(columns={
#     'UF':         'estado',
#     'MUNICIPIO':  'municipio',
#     'LATITUDE':   'latitude',
#     'LONGITUDE':  'longitude',
# }).dropna(subset=['latitude','longitude'])

# regioes_map = { ... }  # igual ao dicionário acima
# df['regiao'] = df['estado'].map(regioes_map)

# Se o arquivo não tiver lat/lon, use geocodificação:
# from geopy.geocoders import Nominatim
# geolocator = Nominatim(user_agent='cgvis')
# ... (geocodifique por município+estado)

print('Célula de adaptação (comentada). Ajuste conforme seu dataset.')

## 5. Download do GeoJSON dos estados e regiões brasileiras

In [ ]:
# GeoJSON dos estados do Brasil (fonte: IBGE simplificado)
GEOJSON_ESTADOS_URL = (
    'https://raw.githubusercontent.com/codeforamerica/click_that_hood/master'
    '/public/data/brazil-states.geojson'
)

resp = requests.get(GEOJSON_ESTADOS_URL, timeout=30)
estados_geojson = resp.json()

# Adicionar região em cada feature
regioes_map_full = {
    'Acre':'Norte','Amazonas':'Norte','Amapá':'Norte','Pará':'Norte',
    'Rondônia':'Norte','Roraima':'Norte','Tocantins':'Norte',
    'Alagoas':'Nordeste','Bahia':'Nordeste','Ceará':'Nordeste',
    'Maranhão':'Nordeste','Paraíba':'Nordeste','Pernambuco':'Nordeste',
    'Piauí':'Nordeste','Rio Grande do Norte':'Nordeste','Sergipe':'Nordeste',
    'Distrito Federal':'Centro-Oeste','Goiás':'Centro-Oeste',
    'Mato Grosso do Sul':'Centro-Oeste','Mato Grosso':'Centro-Oeste',
    'Espírito Santo':'Sudeste','Minas Gerais':'Sudeste',
    'Rio de Janeiro':'Sudeste','São Paulo':'Sudeste',
    'Paraná':'Sul','Rio Grande do Sul':'Sul','Santa Catarina':'Sul',
}
for feat in estados_geojson['features']:
    nome = feat['properties'].get('name', '')
    feat['properties']['regiao'] = regioes_map_full.get(nome, 'Desconhecido')

print(f"GeoJSON carregado: {len(estados_geojson['features'])} estados")

## 6. Paleta de cores por região

In [ ]:
# Cores das regiões (preenchimento dos polígonos dos estados)
COR_REGIAO = {
    'Norte':       '#2E8B57',   # Verde escuro — Amazônia
    'Nordeste':    '#E8A400',   # Âmbar — sertão / litoral
    'Centro-Oeste':'#8B4513',   # Marrom — cerrado
    'Sudeste':     '#1565C0',   # Azul — industrializado
    'Sul':         '#6A0DAD',   # Roxo — frio / gaúcho
    'Desconhecido':'#888888',
}

print('Paleta definida:')
for r, c in COR_REGIAO.items():
    print(f'  {r}: {c}')

## 7. Construção do mapa interativo

O mapa tem **4 camadas** independentes que podem ser ligadas/desligadas pelo controle de camadas (canto superior direito):

| Camada | O que mostra |
|--------|--------------|
| Regiões | Polígonos dos estados coloridos por região; clique em uma região para zoom e isolamento |
| Heatmap | Mapa de calor — densidade de pescadores (azul → amarelo → vermelho) |
| Clusters | Marcadores agrupados; clique para expandir |
| Pontos Brutos | Pontos individuais coloridos por região |

In [ ]:
from folium import GeoJson, GeoJsonTooltip, GeoJsonPopup
from folium.plugins import HeatMap, MarkerCluster, FeatureGroupSubGroup
import folium

# -------------------------------------------------------
# Mapa base
# -------------------------------------------------------
m = folium.Map(
    location=[-14.0, -52.0],
    zoom_start=4,
    tiles='CartoDB positron',
    control_scale=True,
)

# -------------------------------------------------------
# CAMADA 1 — Polígonos dos estados (coloridos por região)
# com clique para zoom e isolamento
# -------------------------------------------------------

# Contagem de pescadores por estado e região
count_estado = df.groupby('estado').size().reset_index(name='total_pescadores')
count_regiao = df.groupby('regiao').size().reset_index(name='total_regiao')

# Enriquecer GeoJSON com contagens
uf_para_nome = {
    'AC':'Acre','AM':'Amazonas','AP':'Amapá','PA':'Pará',
    'RO':'Rondônia','RR':'Roraima','TO':'Tocantins',
    'AL':'Alagoas','BA':'Bahia','CE':'Ceará','MA':'Maranhão',
    'PB':'Paraíba','PE':'Pernambuco','PI':'Piauí',
    'RN':'Rio Grande do Norte','SE':'Sergipe',
    'DF':'Distrito Federal','GO':'Goiás',
    'MS':'Mato Grosso do Sul','MT':'Mato Grosso',
    'ES':'Espírito Santo','MG':'Minas Gerais',
    'RJ':'Rio de Janeiro','SP':'São Paulo',
    'PR':'Paraná','RS':'Rio Grande do Sul','SC':'Santa Catarina',
}
nome_para_uf = {v: k for k, v in uf_para_nome.items()}
total_por_uf = dict(zip(count_estado['estado'], count_estado['total_pescadores']))
total_por_regiao = dict(zip(count_regiao['regiao'], count_regiao['total_regiao']))

for feat in estados_geojson['features']:
    nome_estado = feat['properties'].get('name', '')
    uf = nome_para_uf.get(nome_estado, '')
    regiao = feat['properties'].get('regiao', 'Desconhecido')
    feat['properties']['uf'] = uf
    feat['properties']['pescadores_estado'] = int(total_por_uf.get(uf, 0))
    feat['properties']['pescadores_regiao'] = int(total_por_regiao.get(regiao, 0))

def estilo_estado(feature):
    regiao = feature['properties'].get('regiao', 'Desconhecido')
    cor = COR_REGIAO.get(regiao, '#888888')
    return {
        'fillColor': cor,
        'color': 'white',
        'weight': 1.5,
        'fillOpacity': 0.35,
    }

def highlight_estado(feature):
    return {
        'fillOpacity': 0.75,
        'weight': 3,
        'color': '#FFD700',
    }

# JavaScript para zoom e isolamento ao clicar num estado
click_js = """
function onEachFeature(feature, layer) {
    layer.on('click', function(e) {
        var map = e.target._map;
        map.fitBounds(e.target.getBounds(), {padding: [30, 30]});
    });
}
"""

fg_estados = folium.FeatureGroup(name='🗺️ Regiões / Estados (clique para zoom)', show=True)

GeoJson(
    estados_geojson,
    style_function=estilo_estado,
    highlight_function=highlight_estado,
    tooltip=GeoJsonTooltip(
        fields=['name', 'regiao', 'pescadores_estado', 'pescadores_regiao'],
        aliases=['Estado:', 'Região:', 'Pescadores no Estado:', 'Pescadores na Região:'],
        localize=True,
        sticky=True,
        style='font-size:13px; font-family:sans-serif;',
    ),
    popup=GeoJsonPopup(
        fields=['name', 'regiao', 'pescadores_estado', 'pescadores_regiao'],
        aliases=['<b>Estado</b>', '<b>Região</b>',
                 '<b>Pescadores no Estado</b>', '<b>Pescadores na Região</b>'],
        max_width=300,
    ),
).add_to(fg_estados)

fg_estados.add_to(m)

# -------------------------------------------------------
# CAMADA 2 — Heatmap de densidade de pescadores
# -------------------------------------------------------
fg_heat = folium.FeatureGroup(name='🔥 Heatmap de concentração', show=True)

heat_data = df[['latitude', 'longitude']].dropna().values.tolist()

HeatMap(
    heat_data,
    radius=18,
    blur=22,
    max_zoom=10,
    gradient={
        0.0:  '#0000FF',   # azul frio — poucos pescadores
        0.25: '#00BFFF',
        0.5:  '#00FF00',   # verde
        0.7:  '#FFFF00',   # amarelo
        0.85: '#FF8000',   # laranja
        1.0:  '#FF0000',   # vermelho quente — muitos pescadores
    },
    min_opacity=0.4,
).add_to(fg_heat)

fg_heat.add_to(m)

# -------------------------------------------------------
# CAMADA 3 — MarkerCluster (agrupamento numérico)
# -------------------------------------------------------
fg_cluster = folium.FeatureGroup(name='📍 Marcadores agrupados (MarkerCluster)', show=False)

mc = MarkerCluster(
    options={'maxClusterRadius': 60, 'showCoverageOnHover': False}
)

COR_REGIAO_FOLIUM = {
    'Norte':        'green',
    'Nordeste':     'orange',
    'Centro-Oeste': 'beige',
    'Sudeste':      'blue',
    'Sul':          'purple',
}

# Para performance: amostrar no máximo 3000 pontos no cluster
amostra = df.sample(min(3000, len(df)), random_state=1)
for _, row in amostra.iterrows():
    cor = COR_REGIAO_FOLIUM.get(row['regiao'], 'gray')
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4,
        color=cor,
        fill=True,
        fill_opacity=0.7,
        tooltip=f"{row['municipio']} ({row['estado']}) — {row['regiao']}",
    ).add_to(mc)

mc.add_to(fg_cluster)
fg_cluster.add_to(m)

# -------------------------------------------------------
# CAMADA 4 — Pontos coloridos por região (visíveis em zoom alto)
# -------------------------------------------------------
fg_pontos = folium.FeatureGroup(name='⚫ Pontos individuais por região', show=False)

for regiao, grupo in df.groupby('regiao'):
    cor_hex = COR_REGIAO.get(regiao, '#888888')
    amostra_reg = grupo.sample(min(800, len(grupo)), random_state=42)
    for _, row in amostra_reg.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=3,
            color=cor_hex,
            fill=True,
            fill_color=cor_hex,
            fill_opacity=0.65,
            weight=0.5,
            tooltip=f"{row['municipio']} / {row['estado']}",
        ).add_to(fg_pontos)

fg_pontos.add_to(m)

# -------------------------------------------------------
# Controle de camadas
# -------------------------------------------------------
folium.LayerControl(position='topright', collapsed=False).add_to(m)

# -------------------------------------------------------
# Legenda HTML (canto inferior direito)
# -------------------------------------------------------
legenda_html = """
<div style="
    position: fixed;
    bottom: 40px; right: 10px;
    background: rgba(255,255,255,0.93);
    border: 2px solid #999;
    border-radius: 8px;
    padding: 12px 16px;
    font-family: sans-serif;
    font-size: 13px;
    z-index: 9999;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.25);
">
  <b style="font-size:14px;">Regiões do Brasil</b><br><br>
  <span style="background:#2E8B57;display:inline-block;width:14px;height:14px;border-radius:3px;vertical-align:middle;"></span>&nbsp;Norte<br>
  <span style="background:#E8A400;display:inline-block;width:14px;height:14px;border-radius:3px;vertical-align:middle;"></span>&nbsp;Nordeste<br>
  <span style="background:#8B4513;display:inline-block;width:14px;height:14px;border-radius:3px;vertical-align:middle;"></span>&nbsp;Centro-Oeste<br>
  <span style="background:#1565C0;display:inline-block;width:14px;height:14px;border-radius:3px;vertical-align:middle;"></span>&nbsp;Sudeste<br>
  <span style="background:#6A0DAD;display:inline-block;width:14px;height:14px;border-radius:3px;vertical-align:middle;"></span>&nbsp;Sul<br>
  <br>
  <b>Heatmap</b><br>
  <span style="background:linear-gradient(to right,#0000FF,#00FF00,#FF0000);display:inline-block;width:100px;height:10px;border-radius:3px;"></span><br>
  <span style="font-size:11px;">Baixa &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; Alta</span><br>
  <span style="font-size:11px;color:#555;">concentração de pescadores</span>
</div>
"""
m.get_root().html.add_child(folium.Element(legenda_html))

# -------------------------------------------------------
# Título sobreposto
# -------------------------------------------------------
titulo_html = """
<div style="
    position: fixed;
    top: 12px; left: 50%; transform: translateX(-50%);
    background: rgba(255,255,255,0.92);
    border: 1px solid #bbb;
    border-radius: 8px;
    padding: 8px 20px;
    font-family: sans-serif;
    font-size: 16px;
    font-weight: bold;
    z-index: 9999;
    box-shadow: 1px 1px 5px rgba(0,0,0,0.2);
    text-align: center;
">
  🐟 Distribuição Geográfica de Armadores de Pesca no Brasil<br>
  <span style="font-size:12px;font-weight:normal;color:#555;">Fonte: MPA/SERMOP — Clique em um estado para zoom e isolamento</span>
</div>
"""
m.get_root().html.add_child(folium.Element(titulo_html))

print('Mapa construído com sucesso!')
m

## 8. Salvar o mapa como HTML

In [ ]:
OUTPUT_HTML = 'mapa_pescadores_brasil.html'
m.save(OUTPUT_HTML)
print(f'Mapa salvo em: {OUTPUT_HTML}')

# No Google Colab: baixar o arquivo automaticamente
try:
    from google.colab import files
    files.download(OUTPUT_HTML)
    print('Download iniciado no seu navegador!')
except ImportError:
    print('(Não estamos no Colab — abra o arquivo HTML manualmente no navegador)')

## 9. Estatísticas resumidas (para o relatório)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ---- Gráfico de barras: pescadores por região ----
resumo = df.groupby('regiao').size().reset_index(name='total').sort_values('total', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribuição de Armadores de Pesca no Brasil', fontsize=14, fontweight='bold')

cores_barras = [COR_REGIAO[r] for r in resumo['regiao']]

# Barras por região
axes[0].barh(resumo['regiao'], resumo['total'], color=cores_barras, edgecolor='white')
axes[0].set_xlabel('Número de pescadores registrados')
axes[0].set_title('Por Região')
for i, v in enumerate(resumo['total']):
    axes[0].text(v + 20, i, str(v), va='center', fontsize=10)

# Top 10 estados
top10 = df.groupby(['estado','regiao']).size().reset_index(name='total') \
          .sort_values('total', ascending=False).head(10)
cores_top10 = [COR_REGIAO[r] for r in top10['regiao']]
axes[1].barh(top10['estado'], top10['total'], color=cores_top10, edgecolor='white')
axes[1].set_xlabel('Número de pescadores registrados')
axes[1].set_title('Top 10 Estados')
for i, v in enumerate(top10['total']):
    axes[1].text(v + 5, i, str(v), va='center', fontsize=9)

# Legenda
patches = [mpatches.Patch(color=c, label=r) for r, c in COR_REGIAO.items() if r != 'Desconhecido']
axes[1].legend(handles=patches, title='Região', loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('grafico_resumo_pescadores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo: grafico_resumo_pescadores.png')

## 10. Caption e Conclusão

*(Escreva você mesmo no RELATORIO.md — não use IA para isso)*

### Sugestão de elementos para a legenda (*caption*):

- O mapa exibe a distribuição geográfica dos armadores de pesca registrados no MPA.
- O **heatmap** (mapa de calor) usa gradiente azul→verde→amarelo→vermelho para indicar concentração: regiões vermelhas concentram o maior número de pescadores.
- Os **polígonos dos estados** são coloridos conforme a região geográfica (ver legenda). Ao clicar em um estado, o mapa realiza zoom e isola aquela unidade.
- O controle de camadas (canto superior direito) permite alternar entre heatmap, marcadores agrupados e pontos individuais.

### Sugestão de conclusão não-óbvia para explorar:

- **O litoral do Nordeste e do Sul concentra a maior densidade absoluta de pescadores**, enquanto estados amazônicos (PA, AM) têm alta concentração relativa de pesca fluvial — diferença visível no heatmap ao comparar o padrão costeiro vs. ribeirinho.
- A **Região Norte** registra muitos pescadores em cidades ribeirinhas do interior, revelando que a pesca brasileira não é exclusivamente marítima.
- **São Paulo e Rio de Janeiro** aparecem como grandes polos apesar de não serem estados com vocação pesqueira dominante, sugerindo concentração de armadores (donos de embarcações) em centros econômicos.